In [ ]:
# %pip install mplfinance

In [ ]:
import pandas as pd
import os

def read_taifex(path:str, product: str, product_m_or_w: str):
    '''
        讀取台灣期貨交易所的歷史資料，並選取特定商品代號及月份(週別)的資料

        年月日: str 
        商品代號: str
        商品月份: str

        回傳: DataFrame
    '''
    
    # 讀取 CSV 檔案，並將所有元素轉換成字串
    if os.path.exists(path):
        df = pd.read_csv(path, encoding="BIG5", dtype=str)
    else:
        return pd.DataFrame()
     
    # 移除所有元素的空白
    df = df.map(lambda x: x.strip())

    # 當 '到期月份(週別)' 包含 '/' 時，將 '成交價格' 的值改為 '近月價格' 的值
    df.loc[df['到期月份(週別)'].str.contains('/'), '成交價格'] = df['近月價格']
    
    # 到期月份(週別)中有些資料是202404/202405，代表轉倉，因此分割字串並選取第一個元素並存到新欄位 "商品月份"
    df['商品月份'] = df['到期月份(週別)'].str.split('/').str[0]

    df = df[df["商品代號"] == product]
    df = df[df["商品月份"] == product_m_or_w]

    # 將"交易日期" 和 "交易時間" 合併成一個新的欄位 "交易時間"
    df['Date'] = df['成交日期'] + df['成交時間']

    # 然後，將 "交易時間" 轉換成 datetime 對象
    # 假設 "交易日期" 是 "YYYYMMDD" 的格式，"交易時間" 是 "HHMMSS" 的格式
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d%H%M%S')

    df.set_index('Date', inplace=True)
    df = df.drop(columns=['成交日期'])
    df = df.drop(columns=['成交時間'])

    return df

In [ ]:
import pandas as pd
import os
from datetime import datetime, time
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt
import mplfinance as mpf

# 假設你的 DataFrame 是 df
def to_kline(interval:str, df:pd.DataFrame):
    '''
    將原始交易資料轉換為 K 線資料

    參數:
        interval: K 線的時間間隔，例如 '5min' 代表 5 分鐘
        df: 原始交易資料，必須包含 '成交數量(B+S)' 和 '成交價格' 兩個欄位

    回傳值
        df_kline: K 線資料，包含 'Open', 'High', 'Low', 'Close', 'Volume' 五個欄位
    '''

    df.index = pd.to_datetime(df.index)  # 確保索引是 datetime 對象
    df['成交數量(B+S)'] = df['成交數量(B+S)'].astype(int)
    df['成交價格'] = df['成交價格'].astype(float)

    # 計算 OHLC
    df_ohlc = df['成交價格'].resample(interval).agg({'Open':'first', 'High':'max', 'Low':'min', 'Close':'last'})

    # 計算 Volume
    df_volume = df['成交數量(B+S)'].resample(interval).sum()

    # 將兩個 DataFrame 合併
    df_kline = pd.concat([df_ohlc, df_volume], axis=1)
    df_kline = df_kline.rename(columns={'成交數量(B+S)': 'Volume'})  # 將 '成交數量(B+S)' 重新命名為 'Volume'

    return df_kline

def fetch_time(datetime:datetime, time_start:time=time(8,45), time_end:time=time(13,45), interval:str="5min", symbol:str="MTX"):
    year = datetime.strftime('%Y')
    month = datetime.strftime('%m')
    day = datetime.strftime('%d')

    path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'history_data', 'tw', 'taifex', f'Daily_{year}_{month}_{day}.csv')
    
    # 讀取檔案，如果當天的資料不存在，則讀取下一個月的資料(轉倉後的資料)
    df = read_taifex(path, symbol, year + month)
    if df.empty:
        df = read_taifex(path, symbol, year + (datetime + relativedelta(months=1)).strftime('%m'))
    if df.empty:
        return None
    df_kline = to_kline(interval, df)

    # 轉成固定時間區間
    # df_kline.index = pd.to_datetime(df_kline.index)
    iday = df_kline[df_kline.index.date == datetime.date()]
    iday = iday[iday.index.time >= time_start]
    iday = iday[iday.index.time < time_end]

    return iday

def draw_kline(iday:pd.DataFrame):
    '''
        繪製 K 線圖
        參數:
            iday : pd.DataFrame : 股票資料
        回傳:
            kline_path : str : 圖片檔案路徑
    '''

    mc = mpf.make_marketcolors(up='r',down='g',inherit=True)
    s  = mpf.make_mpf_style(base_mpf_style='tradingview',marketcolors=mc)
    if int(iday.shape[0]) == 0:
        return None, None
    
    # 繪製 K 線圖
    fig, axes = mpf.plot(iday.dropna(), type='candle', style=s,
            ylabel='price',
            volume=True,
            returnfig=True)
    plt.close()

    kline_path = os.path.join("img", "fig.png")
    fig.savefig(kline_path)

    return kline_path

In [ ]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

In [ ]:
def kline_response(kline_path):
    # todo : 這邊要改成從資料庫讀取資料

    if kline_path is None:
        return None

    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", 
                                google_api_key = os.getenv('GEMINI_API_KEY'),)
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": "這是一張小台指的5分k線圖，請根據你對於技術型態的知識，仔細說明這張圖的具體形態，關鍵價格，成交量的變化趨勢，並猜測其隱含意義",
            },  # You can optionally provide text parts
            {"type": "image_url", "image_url": kline_path},
        ]
    )

    response_kline = llm.invoke([message])
    return response_kline.content

In [ ]:
def sig_response(llm, response_kline):
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": f'''
                你是一個厲害的操盤手，是當日沖銷的高手，請透過{response_kline}深入分析接下來一小時應該要作多或是作空，你只能回答 #long 或  #short 或 #unknown。
                ''',
            }, 
        ]
    )

    response = llm.invoke([message])
    return response.content

In [ ]:
# k 線圖

from datetime import datetime, time
import matplotlib.pyplot as plt
import pandas as pd
import os
from datetime import timedelta
import json

# 創建日期範圍
start_date = "2024-02-15"
end_date = "2024-05-30"
date_range = pd.date_range(start=start_date, end=end_date)

# 創建時間範圍  9:00 ~ 13:00
time_range = [time(hour) for hour in range(9,14)]

# 對於每一天的每一小時調用 fetch_time
for date in date_range:
    for start_time in time_range:
        end_time = (datetime.combine(date, start_time) + timedelta(hours=1)).time()
        date_str = date.strftime("%Y%m%d")
        start_time_str = start_time.strftime("%H")
        end_time_str = end_time.strftime("%H")
        formatted_str = f"{date_str}_{start_time_str}_{end_time_str}"
        print(f"date: {date_str}, 從：{start_time_str}～{end_time_str}")
        df = fetch_time(date, start_time, end_time, "5min", "MTX")
        if df is None or len(df)<1:
            continue
        # df.to_csv("temp_data/" + formatted_str + ".csv")
        change = df['Close'].iloc[-1]/df['Open'].iloc[0]
        
        path = draw_kline(df)
        res = kline_response(path)
        sig_str = sig_response(llm, res)
        if sig_str.find("long") > 0:
            sig = 1
        elif sig_str.find("short")>0:
            sig = -1
        else:
            sig = 0
        print(sig_str, sig)

        output_data = {
            "date": date_str,
            "start_time": start_time_str,
            "end_time":end_time_str,
            "model": model,
            "temperature": temperature,
            "top_p": top_p,
            "seed": seed,
            "response_sig": sig,
            "change": change,
            "sig_str":sig_str,
        }
        json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
        out_path = os.path.join("out_kline_202401_05")
        os.makedirs(out_path, exist_ok=True)
        with open(out_path + "/" + formatted_str + ".json", 'w', encoding='utf-8') as f:
            f.write(json_content)
        


In [18]:
import pandas as pd
import json
import os

# 計算準確率和平均報酬率
gain = []
loss = []

tp = 0
fp = 0
tn = 0
fn = 0

files = os.listdir("out_kline_202401_05")
files = sorted(files)

for idx, file in enumerate(files):
    if file.split("_")[1] != "09":
        continue
    # print(file)
    with open("out_kline_202401_05/" + file, "r", encoding='utf-8') as f:
        data = json.load(f)
        sig = data["response_sig"]
        print(data["response_sig"], data["sig_str"])

    if idx+1 == len(files):
        break
    # print(files[idx+1])
    with open("out_kline_202401_05/" + files[idx+1], "r", encoding='utf-8') as f:
        data = json.load(f)
        rtn = data["change"]
        rtn = float(rtn) - 1
        print(data["change"])
    print()
    
    if pd.isna(rtn):
        rtn = 0

    if sig > 0:
        if rtn > 0:
            tp += 1
            gain.append(rtn)
        else:
            fp += 1
            loss.append(rtn)
    elif sig == -1: 
        if rtn < 0:
            tn += 1
            gain.append(-rtn)
        else:
            fn += 1
            loss.append(-rtn)



accuracy = (tp + tn) / (tp + fp + tn + fn)
print(f'Accuracy: {accuracy}')
print(f'Average gain: {sum(gain) / len(gain)}')
print(f'Average loss: {sum(loss) / len(loss)}')
print(f'expected value: {(sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)}')
# print(tn+tp+fn+fp)

1 #long 

0.9976419066868789

0 #unknown 

0.996815829874339

-1 #short 

0.9996582365003418

1 #long 

0.998747081268865

1 #long 

0.9982993197278912

0 #unknown 

0.9983535823776541

1 #long 

1.0000572082379862

1 #long 

1.0010853421683994

1 #long 

1.0018255462376633

1 #long 

1.0042764283270613

0 #unknown 

1.0002880184331797

0 #unknown 

0.9979748886188741

-1 #short 

0.9992463331207606

0 #unknown 

1.0010808350873202

1 #long 

0.9968008082168716

-1 #short 

0.9998879677347076

0 #unknown 

0.9997210910916494

0 #unknown 

0.9982215305952314

0 #unknown 

1.0013898154325105

1 #long 

1.000441940117114

0 #unknown 

這個圖表顯示了短期的價格波動，很難根據它來判斷接下來的趨勢。雖然買盤力量在 09:50 反彈，但這並不一定表示價格會繼續上升。市場情緒可能會再次轉變，價格也可能繼續下跌。

你需要更多資訊，例如更長期的圖表、經濟數據、新聞事件等，才能做出更準確的判斷。

0.9983441881002318

-1 #short 

0.9992752815252537

-1 #short 

1.0

0 #unknown 

雖然圖表顯示開盤跳空向上，並出現多根綠色K線，表明市場處於上漲趨勢。但成交量萎縮，顯示市場的買盤力量正在減弱，上漲動能可能不足。

**僅憑5分鐘K線圖無法做出確定的交易決策。** 

需要更多資訊，例如：

* 更長時間的K線圖，例如15分鐘、30分鐘或小時K線圖。
* 技術指標，例如MACD、R